# Tissue-level junction support and coverage for candidate events, case III

R notebook that summarizes GTEx tissue-level split read support of splice junctions and read coverage of constitutive exons surrounding the ATE.

Short workflow:
  - load candidate events list
  - load per-sample surrounding exons coverage (generated by bash script for our candidate events)
    - summarize to per-tissue coverage
  - load per-sample split read support of splice junctions (generated by bash script for our candidate events)
    - summarize to per-tissue counts
  - join the junction support and coverage data 
    - save to intermediate data file `cov_junc_tis_nJ.novel_junc_ATEPE.n260.RData`

In [ ]:
library(dplyr)
library(tidyr)
library(purrr) #map_dfr
library(stringr)
library(stringi)

library(ggplot2)
library(latex2exp)
#for large tables
library(data.table)
library(dtplyr)
setDTthreads(4)

In [ ]:
setwd("ATEPE_from_novel_junc_data")#in novel_junc folder
dat_dir = "../../references/"
pictures_dir = "."

In [ ]:
gtex_cols_raw <- read.table(paste(dat_dir,"GTEX_polyA/GTEX_SMTS_SMTSD_color_codes.tsv",sep="/"),
                        sep="\t",comment.char="",header = TRUE,stringsAs=FALSE) #"/gss/mvlasenok/"
gtex_cols <- gtex_cols_raw %>% 
    select(rcolor,SMTS) %>% 
    unique() %>% group_by(SMTS) %>% summarise(rcolor=rcolor[1]) %>% 
    mutate(SMTS=gsub(SMTS,pattern = " ",replacement = "_")) %>% 
    mutate(SMTS=gsub(x=SMTS,pattern = "Blood_Vessel",repl="Artery")) %>% 
    mutate(SMTS=gsub(SMTS,pattern = "_Tissue",replacement = "")) %>% 
    with(.,setNames(as.character(rcolor), SMTS))
gtex_cols <- c(gtex_cols,Whole_Blood="magenta",
               Bladder="chocolate",Minor_Salivary_Gland="wheat3",Spleen="burlywood3",Small_Intestine="peachpuff3",
               Cervix='plum2')
gtex_smtsd <- gtex_cols_raw %>% select(SMTSD,rcolor,rcolor_brain) %>% 
mutate(SMTSD=gsub(x=SMTSD,pattern = " - ",replacement = '-'),
       SMTSD=gsub(x=SMTSD,pattern = " ",replacement = '_')) %>% 
 with(.,setNames(as.character(rcolor), SMTSD))
gtex_smtsd <- c(gtex_smtsd,'Colon-Sigmoid'="burlywood2",Bladder="chocolate",
                 'Small_Intestine-Terminal_Ileum'="burlywood3",Spleen="burlywood3",
                 'Cervix-Ectocervix'='plum2','Cervix-Endocervix'='plum2',
                 'Esophagus-Gastroesophageal_Junction'="burlywood4",Minor_Salivary_Gland="wheat3")

gtex_smtsd["Brain-Cerebellar_Hemisphere"] <- "yellow4"
gtex_smtsd["Brain-Cerebellum"] <- "yellow4" 

gtex_cols["Cerebellum"] <- "yellow4" 

In [ ]:
theme_blank <- theme_bw() +
    theme(panel.grid = element_blank(),
        strip.background = element_rect(fill = NA))
        #legend.position = "right")

# ATE-PE from nJ data from tissues

In [ ]:
ate <- read.table("novel_junc_ATEPE.n260.skip.tsv",head=T,stringsAsFact=FALSE)

ate <- ate %>% 
mutate(chr = str_split_fixed(ex,pat="_",n=4)[,1],
       strand = str_split_fixed(ex,pat="_",n=4)[,4],
       ss1=ifelse(strand=="+",str_split_fixed(E,pat="_",n=2)[,1],ss1.junc),
       ss4=ifelse(strand=="+",ss4.junc,str_split_fixed(E,pat="_",n=2)[,2]),
       ss2=ifelse(strand=="+",str_split_fixed(ex,pat="_",n=4)[,2],ss4.junc),
       ss3=ifelse(strand=="+",ss1.junc,str_split_fixed(ex,pat="_",n=4)[,3])) %>%   
mutate(I1=paste(chr,ss1,ss2,strand,sep="_"),I2=paste(chr,ss3,ss4,strand,sep="_"),
       E=paste(chr,ss1,ss4,strand,sep="_"),
       ex=paste(chr,ss2,ss3,strand,ss1,ss4,sep="_"),.keep="unused")  %>% 
#switch I1,I2 for - strand
mutate(I1t=I1,I1=ifelse(grepl(ex,pat="+",fixed=T),I1,I2),I2=ifelse(grepl(ex,pat="+",fixed=T),I2,I1t)) %>% 
dplyr::select(-I1t)

add gene_name:

In [ ]:
gene_names <- read.table("../../references/chess3.1.3.GRCh38.prot_cod_trid-gene_name.tsv",stringsAsFactors=F,
                         col.names = c("transcript_id","gene_name")) 
#"~/Documents/arcuda_gss/mvlasenok/references/chess3.1.3.GRCh38.prot_cod_trid-gene_name.tsv"
gene_names <- setNames(gene_names$gene_name,nm = gene_names$transcript_id)
ate <- ate %>% rowwise %>% 
mutate(tr_id1=str_split(tr_id,pat=",",simplify = T)[1]) %>% ungroup %>% 
mutate(gene_name=gene_names[tr_id1],.keep="unused")

Samples

In [ ]:
samples <- read.csv(
    "../../references/GTEX/GTEX_SraRunTable_w_sample_id.txt",stringsAsFactors = F, 
    header=T,col.names= c("sample","sample_gtexid","tissue","SMTSD")) %>% 
#For smtsd 
    mutate(tissue=ifelse(SMTSD=="",tissue,paste(tissue,SMTSD,sep = "-")))

## Load data from tissues :

### Surrounding exons coverage

In [ ]:
sur_ex <- read.table("ATE_PE_e1_e2.PE_id.n260.bed",stringsAsFact=F,
        col.names = c("chr","start","end","name","score","strand")) %>% 
    mutate(name=gsub(x=name,pat="+_",fixed=T,rep="+|"),name=gsub(x=name,pat="-_",fixed=T,rep="-|"),
        name=gsub(x=name,pat="_e",fixed=T,rep="|e")) 
#e1 and e2 are not corrected for the strand. e2 has bigger coords, but not necessarily is downstream

tsv generated by script `sbatch_chess_surex_coverage_bysample.sh` from GTEx sample bigwigs and custom parsing script

In [ ]:
cov_sur <- fread("ATE_PE_sur_ex_cov.n260.tsv",header = T,stringsAsFactors = F,quote="\'") %>% 
    pivot_longer(cols=-c(chr,start,end),names_to = c("sample",NA),names_sep = fixed(".b"),values_to = "cov")
    cov_sur <- cov_sur %>% 
    lazy_dt() %>% 
    left_join(lazy_dt(sur_ex[,1:4]),.,relationship = "many-to-many") %>% 
    left_join(.,samples[c("sample","tissue")]) %>% 
    unique 

 group by smtsd

In [ ]:
cov_sur_tis <- cov_sur %>% as_tibble() %>% 
group_by(chr,start,end,name,tissue) %>% 
summarise(coverage=mean(cov,na.rm=T),.groups = "drop") %>% 
mutate(ex=str_split_fixed(name,n=2,pat=fixed("|"))[,1],
     E=str_split_fixed(name,n=3,pat=fixed("|"))[,2],
     sur_ex=factor(str_split_fixed(name,n=3,pat=fixed("|"))[,3]),.keep="unused") 

cov_sur_tis <- cov_sur_tis %>% 
select(-c(chr,start,end)) %>% 
pivot_wider(names_from=sur_ex, values_from = coverage,values_fill = 0) %>% 
rename(E_junc=E) %>% 
#exchange e1 and e2 for minus strand
mutate(e1t=e1,
       e1=ifelse(grepl(ex,pat="+",fixed=T),e1,e2),
       e2=ifelse(grepl(ex,pat="+",fixed=T),e2,e1t)) %>% select(-e1t)

In [ ]:
#plot e1 and e2 distribution
#options(repr.plot.width = 5, repr.plot.height = 3, repr.plot.res = 150)
#ggplot(cov_sur_tis,aes(x=log(e2/e1)))+geom_histogram(bins=50)+theme_blank

### junction counts
tsv generated by `sbatch_chess_junction_counts.sh` from IPSA output

In [ ]:
split_read <- fread("junction_counts.n260.per_sample.tsv", col.names = c("junc","reads","sample"))
ate_junc <- ate %>% 
    select(ex,I1,I2,E) %>% unique %>%  
    #mutate(ex=paste(x=ex,gsub(E,pat="chr[[:digit:]XYM]+_|_[-+]",rep=""),sep="|")) %>%  
    mutate(ex=gsub(x=ex,pat="+_",fixed=T,rep="+|"),ex=gsub(x=ex,pat="-_",fixed=T,rep="-|")) %>% 
    pivot_longer(-ex,values_to="junc")
split_read <- split_read %>% #head(n=100) %>% 
    mutate(
        E_junc=stri_split_fixed(ex,pattern = fixed("|"),n=2,simplify = T)[,2],
        ex=str_split_fixed(ex,pat=fixed("|"),n=2)[,1],
        E_junc=paste(str_split_fixed(ex,pat="_",n=4)[,1],E_junc,str_split_fixed(ex,pat="_",n=4)[,4],sep="_"))

 group by smtsd

In [ ]:
split_read_tis <- split_read %>% 
    lazy_dt() %>% 
    mutate(sample=gsub(x=sample,pat=".?J6/?",rep="")) %>% 
    left_join(.,samples[c("sample","tissue")]) %>% 
    as_tibble() %>% 
    group_by(junc,tissue) %>% summarize(reads=sum(reads)) %>% ungroup

In [ ]:
split_read_tis <- left_join(tibble(ate_junc),split_read_tis,relationship = "many-to-many") %>%  
    select(-junc) %>% 
    pivot_wider(values_from=reads,values_fill = 0) %>% 
    mutate(E_junc=str_split_fixed(ex,pat=fixed("|"),n=2)[,2],
        ex=str_split_fixed(ex,pat=fixed("|"),n=2)[,1],
        E_junc=paste(str_split_fixed(ex,pat="_",n=4)[,1],E_junc,str_split_fixed(ex,pat="_",n=4)[,4],sep="_")) 

In [ ]:
#plot junc counts distribution
#options(repr.plot.width = 4, repr.plot.height = 2.5, repr.plot.res = 200)
#split_read_tis %>% filter(I1+I2>=10) %>% 
#ggplot(.,aes(x=I1/(I1+I2)))+geom_histogram(bins=50)+
#theme_blank

### join e1,e2 with junction counts

In [ ]:
cov_junc_tis_nJ <- split_read_tis %>% filter(!is.na(tissue)) %>% 
    mutate(E_junc=gsub(E_junc,pat="chr[0-9MXY]+_|_[+-]",rep="")) %>% 
full_join(cov_sur_tis) %>% 
    mutate(across(c(I1,I2,E,e1,e2),~ifelse(is.na(.x),0,.x))) 

In [ ]:
#plot tau and \psi1 distributions
#ggplot(cov_junc_tis_nJ,aes(x=log(e2/e1)))+geom_histogram(bins=50)+theme_blank
#ggplot(cov_junc_tis_nJ,aes(x=I1/(I1+I2)))+
#geom_histogram(binwidth = 0.05,fill="grey50")+#fill=brewer.pal("Greens",n=4)[4]) +
#xlab(TeX("I_1/(I_1+I_2)"))+
#theme_blank
#ggsave(filename = "F1_E_caseIII.pdf", device = "pdf", path = pictures_dir, width=p_width,height=p_height)

Add exon annotation data

In [ ]:
#add ATE-junc annotattion chracteristics
ate_short <- ate %>%
    mutate(
        ex = gsub(x = ex, pat = "+_", fixed = T, rep = "+|"),
        ex = gsub(x = ex, pat = "-_", fixed = T, rep = "-|"),
        ex = str_split_fixed(ex, pat = fixed("|"), n = 2)[, 1]) %>%
    mutate(E_junc = gsub(E, pat = "chr[0-9MXY]+_|_[+-]", rep = ""), .keep = "unused") %>%
    select(
        ex, E_junc, stop_cod, gene_name, reads, within_ex,
        UTR_len, tr_id, ss1.ate, ss4.ate, ss1.junc, ss4.junc) %>% unique
cov_junc_tis_nJ <- left_join(cov_junc_tis_nJ, ate_short)

In [ ]:
save(cov_junc_tis_nJ, file = "cov_junc_tis_nJ.novel_junc_ATEPE.n260.RData")#cov_junc_tis_nJ